In [1]:
import talib
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import numpy as np
from plotly.subplots import make_subplots
import backtesting
from backtesting import Backtest, Strategy


In [2]:
df = yf.download("BTC-USD", start='2021-01-01', end='2023-06-02', interval='1d').drop(columns=['Adj Close'])
df = df.reset_index()
df.columns = df.columns.str.lower()

[*********************100%%**********************]  1 of 1 completed


In [3]:


# Рассчитываем TEMA и MACD
df['tema'] = talib.TEMA(df['close'], timeperiod=24)
df['macd'], df['macd_signal'], df['macd_hist'] = talib.MACD(df['close'], fastperiod=12, slowperiod=26, signalperiod=9)
df = df.dropna()
# Создаем сигналы для покупки и продажи
df['signal'] = 0
df.loc[(df['macd'] > df['macd_signal']) & (df['close'] > df['tema']), 'signal'] = 1  # Сигнал на покупку
df.loc[(df['macd'] < df['macd_signal']) & (df['close'] < df['tema']), 'signal'] = -1  # Сигнал на продажу

# Создаем график с двумя подграфиками
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[0.7, 0.3])

# График цены закрытия и TEMA
fig.add_trace(go.Scatter(x=df.index, y=df['close'], name='Цена закрытия', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['tema'], name='TEMA', line=dict(color='orange')), row=1, col=1)

# Добавляем сигналы покупки и продажи
buy_signals = df[df['signal'] == 1]
sell_signals = df[df['signal'] == -1]
no_signals = df[df['signal'] == 0]

fig.add_trace(go.Scatter(x=buy_signals.index, y=buy_signals['close'], mode='markers', 
                         name='Покупка', marker=dict(symbol='triangle-up', size=10, color='green')), row=1, col=1)
fig.add_trace(go.Scatter(x=sell_signals.index, y=sell_signals['close'], mode='markers', 
                         name='Продажа', marker=dict(symbol='triangle-down', size=10, color='red')), row=1, col=1)
fig.add_trace(go.Scatter(x=no_signals.index, y=no_signals['close'], mode='markers', 
                         name='Нет сигнала', marker=dict(symbol='circle', size=10, color='gray')), row=1, col=1)
# График MACD и сигнальной линии
fig.add_trace(go.Scatter(x=df.index, y=df['macd'], name='MACD', line=dict(color='blue')), row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['macd_signal'], name='Сигнальная линия', line=dict(color='red')), row=2, col=1)

# Настройка макета графика
fig.update_layout(height=800, title='Стратегия MACD TEMA Crossover')
fig.update_xaxes(title_text='Дата', row=2, col=1)
fig.update_yaxes(title_text='Цена', row=1, col=1)
fig.update_yaxes(title_text='MACD', row=2, col=1)

fig.show()


In [4]:
# Добавим столбец где будет указан id трейда
df['action_id'] = 0
df['action_id'] = df['signal'].ne(df['signal'].shift()).cumsum()
# Разделим датафрейм на обучающую и тестовую выборки
split_value = int(len(df) * 0.7)
train_df = df.iloc[:split_value]
test_df = df.iloc[split_value:].reset_index(drop=True)


#Теперь  сделаем бэктест стратегии используя библиотеку backtesting
# для этого нам нужно создать класс стратегии

class TemaMacdStrategy(Strategy):
    def init(self):
        self.signal = self.I(lambda: self.data.Signal)
        self.previous_signal = 0

    def next(self):
        current_signal = self.signal[-1]

        if current_signal != self.previous_signal:
            if current_signal == 1:
                if self.position.is_short:
                    self.position.close()
                    
                if not self.position.is_long:
                    self.buy()
                    
            elif current_signal == -1:
                if self.position.is_long:
                    self.position.close()
                   
                if not self.position.is_short:
                    self.sell()
                    
            elif current_signal == 0:
                if self.position:
                    self.position.close()
                    

        self.previous_signal = current_signal

# Подготовка данных для бэктестинга
bt_df = test_df.copy()
bt_df.columns = bt_df.columns.str.capitalize()
bt_df.rename(columns={'Date': 'Datetime'}, inplace=True)
bt_df["Datetime"] = pd.to_datetime(bt_df["Datetime"])
bt_df.set_index('Datetime', inplace=True)

# Создаем объект класса Backtest
bt = Backtest(bt_df, TemaMacdStrategy, cash=1000000, commission=.002, exclusive_orders=True)

# Запускаем бэктест
stats = bt.run()

# Выводим статистику
print(stats[:27])

# выводим график
bt.plot(
    plot_equity=True,
    plot_drawdown=True,
    relative_equity=False,
)



Start                     2022-10-01 00:00:00
End                       2023-06-01 00:00:00
Duration                    243 days 00:00:00
Exposure Time [%]                   78.278689
Equity Final [$]               1518817.818289
Equity Peak [$]                1566515.708914
Return [%]                          51.881782
Buy & Hold Return [%]               38.876552
Return (Ann.) [%]                   86.859271
Volatility (Ann.) [%]                79.67726
Sharpe Ratio                         1.090139
Sortino Ratio                        3.732636
Calmar Ratio                         6.027924
Max. Drawdown [%]                  -14.409483
Avg. Drawdown [%]                   -5.134963
Max. Drawdown Duration       66 days 00:00:00
Avg. Drawdown Duration       19 days 00:00:00
# Trades                                   26
Win Rate [%]                        46.153846
Best Trade [%]                       23.77219
Worst Trade [%]                     -6.634028
Avg. Trade [%]                    

c:\Users\ma2se\OneDrive\Documents\VS_CODE_PROJECTS\notebooks\.venv\Lib\site-packages\backtesting\_plotting.py:456: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



Row(id='1524', ...)

In [5]:
# групируем по id осталяя только первые строки каждой сделки - это наши точки смены сигнала
train_df = train_df.groupby('action_id').first().reset_index()

# добавим столбец с профитом
train_df['profit'] = 0.0

# Смещаем столбцы 'close' и 'signal', чтобы получить данные следующего бара
train_df['next_close'] = train_df['close'].shift(-1)
train_df['next_signal'] = train_df['signal'].shift(-1)

# Рассчитываем PnL
train_df['profit'] = (train_df['next_close'] - train_df['close']) * train_df['signal']

# Заполняем PnL значением 0 для последнего бара, так как для него нет следующего бара
train_df['profit'].fillna(0, inplace=True)

# Удаляем вспомогательные столбцы
train_df.drop(columns=['next_close', 'next_signal'], inplace=True)

# Добавляем целевую переменную : 0 - cделка принесла убыток, 1- сделка принесла прибыль
train_df['Label'] = train_df['profit'].apply(lambda x: 1 if x >= 0 else 0)

# И оставляем только те строки где мы входили в позицию
train_df = train_df[train_df['signal'] != 0]

C:\Users\ma2se\AppData\Local\Temp\ipykernel_19204\634049111.py:15: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.





In [6]:

import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier


# Разделяем признаки и метки
X_train = train_df[['tema', 'macd', 'macd_signal', 'close', 'signal']]
y_train = train_df['Label']

scaler = StandardScaler()      
x_scaled = scaler.fit_transform(X_train)

#обучаем модель
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(x_scaled, y_train)

# Важность признаков
feature_importance = pd.Series(model.feature_importances_, index=X_train.columns)
print("Feature Importance:\n", feature_importance.sort_values(ascending=False))

Feature Importance:
 macd           0.253937
tema           0.247644
close          0.236121
macd_signal    0.227951
signal         0.034348
dtype: float64


In [7]:
# повторяем все тоже самое с данными для теста
test_df = test_df.groupby('action_id').first().reset_index()
test_df['next_close'] = test_df['close'].shift(-1)
test_df['next_signal'] = test_df['signal'].shift(-1)
test_df['profit'] = (test_df['next_close'] - test_df['close']) * test_df['signal']
test_df['profit'].fillna(0, inplace=True)
test_df['Label'] = test_df['profit'].apply(lambda x: 1 if x >= 0 else 0)
test_df = test_df[test_df['signal'] != 0]
#test_df[["date", "close", "signal", "action_id", "profit", "Label"]].head(20)


C:\Users\ma2se\AppData\Local\Temp\ipykernel_19204\2087982281.py:6: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.





In [8]:
# Для предсказания используем данные в столбацах ниже
X_test = test_df[['tema', 'macd', 'macd_signal', 'close', 'signal']]
y_test = test_df['Label']

# делаем предсказание
y_pred = model.predict(X_test)
# описание наших метрик
# accuracy - точность нашей модели
# precision - точность предсказания
# recall - полнота предсказания
# f1-score - среднее гармоническое точности и полноты
# support - количество образцов в каждом классе
# выводим предсказание
print(y_pred)
# выведем нашу точность
print(accuracy_score(y_test, y_pred))
# выведем нашу матрицу
print(confusion_matrix(y_test, y_pred))
# выведем нашу классификационную отчет
print(classification_report(y_test, y_pred))

[1 1 1 1 1 1 0 0 1 1 1 1 1 1 0 0 0 0 1 0 0 0 0 0 1 1]
0.4230769230769231
[[5 9]
 [6 6]]
              precision    recall  f1-score   support

           0       0.45      0.36      0.40        14
           1       0.40      0.50      0.44        12

    accuracy                           0.42        26
   macro avg       0.43      0.43      0.42        26
weighted avg       0.43      0.42      0.42        26



c:\Users\ma2se\OneDrive\Documents\VS_CODE_PROJECTS\notebooks\.venv\Lib\site-packages\sklearn\base.py:486: UserWarning:

X has feature names, but RandomForestClassifier was fitted without feature names



In [9]:
test_df['pred'] = y_pred
test_df["date"] = pd.to_datetime(test_df["date"])
test_df = test_df.set_index("date")
bt_df['pred'] = test_df['pred']
bt_df['pred'] = bt_df['pred'].ffill()
# зануляем сделки предсказанные как убыточные
bt_df.loc[(bt_df['pred'] == 0) & (bt_df['Signal'] != 0), 'Signal'] = 0

In [10]:
bt = Backtest(bt_df, TemaMacdStrategy, cash=1000000, commission=.002, exclusive_orders=True)

# Запускаем бэктест
stats = bt.run()

# Выводим статистику
print(stats[:27])



# выводим график
bt.plot(
    plot_equity=True,
    plot_drawdown=True,
    relative_equity=False,
)


Start                     2022-10-01 00:00:00
End                       2023-06-01 00:00:00
Duration                    243 days 00:00:00
Exposure Time [%]                   39.344262
Equity Final [$]                1357899.96825
Equity Peak [$]                1400394.452625
Return [%]                          35.789997
Buy & Hold Return [%]               38.876552
Return (Ann.) [%]                     58.0364
Volatility (Ann.) [%]               40.569064
Sharpe Ratio                         1.430558
Sortino Ratio                         4.98473
Calmar Ratio                         6.112177
Max. Drawdown [%]                   -9.495209
Avg. Drawdown [%]                    -3.29955
Max. Drawdown Duration      100 days 00:00:00
Avg. Drawdown Duration       30 days 00:00:00
# Trades                                   15
Win Rate [%]                             40.0
Best Trade [%]                       23.77219
Worst Trade [%]                      -3.11043
Avg. Trade [%]                    

c:\Users\ma2se\OneDrive\Documents\VS_CODE_PROJECTS\notebooks\.venv\Lib\site-packages\backtesting\_plotting.py:456: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



Row(id='2881', ...)